# V9 PIR Pipeline: The 'Recall Max' Engine
Multi-Source Candidate Generation (50+ Analytical Insights) + GPU Training + 3-Fold Robust CV
Targeting Precision@10 > 0.25


In [1]:
import polars as pl
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix
import optuna
import gc
import warnings
import re
warnings.filterwarnings('ignore')

# Set Seeds for Reproducibility
SEED = 42
np.random.seed(SEED)

T_PATH = '/kaggle/input/datasets/kinonquc/qkindataset2/transaction_full_2025.parquet'
I_PATH = '/kaggle/input/datasets/kinonquc/qkindataset2/items.parquet'


In [2]:
print("=== LOADING & CLEANING DATA ===")
df_raw = pl.read_parquet(T_PATH).select([
    pl.col('customer_id').cast(pl.Int64),
    pl.col('item_id').cast(pl.Utf8),
    pl.col('updated_date').cast(pl.Datetime).alias('event_ts'),
    pl.col('location').cast(pl.Utf8),
    pl.col('price').cast(pl.Float32),
    pl.col('quantity').cast(pl.Float32)
]).with_columns(pl.col('event_ts').dt.month().alias('month'))

items_df = pl.read_parquet(I_PATH).select([
    'item_id', 'category', 'category_l1', 'category_l2', 'category_l3', 'brand', 'size'
]).with_columns(pl.col('item_id').cast(pl.Utf8))

cat_cols = ['category', 'category_l1', 'category_l2', 'category_l3', 'brand']
for c in cat_cols:
    items_df = items_df.with_columns(pl.col(c).fill_null('Unknown'))
    # Cap at 254 unique values for GPU compatibility (Idea 35/43)
    top_vals = items_df[c].value_counts().sort('count', descending=True).head(254)[c].to_list()
    items_df = items_df.with_columns(
        pl.when(pl.col(c).is_in(top_vals)).then(pl.col(c)).otherwise(pl.lit('Other')).alias(c)
    )
    items_df = items_df.with_columns(pl.col(c).cast(pl.Categorical).to_physical().cast(pl.Int32).alias(f'{c}_id'))

# Item Age Proxy (from Idea 34/45)
def standardize_age(val):
    if not isinstance(val, str): return -1.0
    val = val.lower().strip()
    match = re.search(r'(\d+)\s*(m|y|tháng|tuổi)', val)
    if match:
        num = float(match.group(1))
        unit = match.group(2)
        if unit in ['m', 'tháng']: return num / 12.0
        return num
    return -1.0

size_map = {row[0]: standardize_age(row[1]) for row in items_df.select(['item_id', 'size']).iter_rows()}
items_df = items_df.with_columns(pl.col('item_id').replace(size_map, default=-1.0).cast(pl.Float32).alias('item_age_proxy'))


=== LOADING & CLEANING DATA ===


In [3]:
print("=== V9 RETRIEVER: MULTI-SOURCE FUNNEL ===")
class V9Retriever:
    def __init__(self, history_df, items_df):
        self.history_df = history_df
        self.items_df = items_df
        self.max_ts = history_df['event_ts'].max()
        
        # 1. Global & Local Popularity (Idea 30)
        print(" Building Popularity & Hub Indexes...")
        self.item_locs = history_df.group_by('item_id').agg(pl.col('location').unique().alias('item_hubs'))
        self.item_prcs = history_df.group_by('item_id').agg(pl.col('price').median().alias('item_p'))
        
        self.global_top = history_df.filter(pl.col('event_ts') >= self.max_ts - pl.duration(days=14))\
            .group_by('item_id').len().sort('len', descending=True).head(150).select('item_id')
            
        self.local_heroes = history_df.filter(pl.col('event_ts') >= self.max_ts - pl.duration(days=60))\
            .group_by(['location', 'item_id']).len()\
            .sort(['location', 'len'], descending=[False, True])\
            .group_by('location').head(80)

        # 2. CF Indexes (SVD + I2I)
        self._build_cf()
        
        # 3. Size Ladder (Idea 34)
        self._build_size_ladder()
        
        # 4. Replenishment (Idea 6/44)
        self._build_replenishment()

    def _build_cf(self):
        print(" Building SVD (160) & I2I Matrices...")
        hist = self.history_df.group_by(['customer_id', 'item_id']).agg([
            pl.col('quantity').sum().alias('w'),
            pl.col('event_ts').max().alias('last_ts')
        ])
        # Time-decay weights
        hist = hist.with_columns((pl.col('w') * (0.9 ** ((self.max_ts - pl.col('last_ts')).dt.total_days() / 30.0))).alias('w'))
        
        hist = hist.with_columns([
            pl.col('customer_id').rank('dense').cast(pl.Int64).alias('u_idx') - 1,
            pl.col('item_id').rank('dense').cast(pl.Int32).alias('i_idx') - 1
        ])
        self.u_map = hist.select(['customer_id', 'u_idx']).unique()
        self.i_map = hist.select(['item_id', 'i_idx']).unique()
        self.u2idx = dict(zip(self.u_map['customer_id'], self.u_map['u_idx']))
        self.items_list = self.i_map.sort('i_idx')['item_id'].to_list()
        
        self.matrix = csr_matrix((hist['w'].to_numpy(), (hist['u_idx'].to_numpy(), hist['i_idx'].to_numpy())), 
                                shape=(self.u_map.height, self.i_map.height), dtype=np.float32)
        
        self.svd = TruncatedSVD(n_components=160, random_state=SEED)
        self.u_vecs = self.svd.fit_transform(self.matrix)
        self.i_vecs = self.svd.components_.T
        
        norm_m = normalize(self.matrix, norm='l2', axis=0)
        self.i2i_sim = (norm_m.T.dot(norm_m)).astype(np.float32)
        self.i2i_sim.setdiag(0)

    def _build_size_ladder(self):
        # Maps current item to next age bucket item in the same category
        print(" Building Size Ladder...")
        # Simple logic: for each category, find the items sorted by age_proxy
        self.ladder_map = self.items_df.filter(pl.col('item_age_proxy') > 0).sort(['category_l3', 'item_age_proxy'])

    def _build_replenishment(self):
        print(" Building Replenishment Cycles...")
        # Items with repeat patterns (Idea 44)
        repeats = self.history_df.sort(['customer_id', 'item_id', 'event_ts'])\
            .with_columns(pl.col('event_ts').diff().dt.total_days().over(['customer_id', 'item_id']).alias('gap'))\
            .filter(pl.col('gap').is_not_null())
        self.item_gaps = repeats.group_by('item_id').agg(pl.col('gap').median().alias('m_gap'))

    def get_candidates(self, target_users):
        hist_s = self.history_df.filter(pl.col('customer_id').is_in(target_users))
        cands = {}
        
        # 1. Repurchase (History)
        cands['rep'] = hist_s.select(['customer_id', 'item_id']).unique()
        
        # 2. Collaborative (SVD & I2I)
        t_idx = [self.u2idx[u] for u in target_users if u in self.u2idx]
        t_u = [u for u in target_users if u in self.u2idx]
        i_arr = np.array(self.items_list)
        chunk = 2000
        c_i2i, c_svd = [], []
        for i in range(0, len(t_idx), chunk):
            idx = t_idx[i:i+chunk]
            u_b = np.array(t_u[i:i+chunk])
            # SVD (Latent)
            s_s = self.u_vecs[idx].dot(self.i_vecs.T)
            t60 = np.argsort(-s_s, axis=1)[:, :60]
            c_svd.append(pl.DataFrame({'customer_id': pl.Series(np.repeat(u_b, 60), dtype=pl.Int64), 'item_id': i_arr[t60.flatten()]}))
            # I2I (Co-purchase)
            s_i = self.matrix[idx].dot(self.i2i_sim).toarray()
            t80 = np.argsort(-s_i, axis=1)[:, :80]
            mask = np.take_along_axis(s_i, t80, axis=1) > 0
            c_i2i.append(pl.DataFrame({'customer_id': pl.Series(np.repeat(u_b, 80)[mask.flatten()], dtype=pl.Int64), 
                                      'item_id': i_arr[t80.flatten()][mask.flatten()]}))
        
        cands['svd'] = pl.concat(c_svd).unique() if c_svd else pl.DataFrame(schema={'customer_id': pl.Int64, 'item_id': pl.Utf8})
        cands['i2i'] = pl.concat(c_i2i).unique() if c_i2i else pl.DataFrame(schema={'customer_id': pl.Int64, 'item_id': pl.Utf8})
        
        # 3. Trends (Global + Local)
        cands['global'] = pl.DataFrame({'customer_id': target_users}).join(self.global_top.with_columns(pl.lit(1).alias('_k')), how='cross').drop('_k')
        user_loc = hist_s.group_by('customer_id').agg(pl.col('location').mode().first().alias('location'))
        cands['local'] = user_loc.join(self.local_heroes, on='location').select(['customer_id', 'item_id']).unique()
        
        # 4. Replenishment (Idea 6)
        # Find items bought by user that are 'due'
        due_items = hist_s.join(self.item_gaps, on='item_id', how='inner')\
            .with_columns((self.max_ts - pl.col('event_ts').max()).dt.total_days().over(['customer_id', 'item_id']).alias('recency'))\
            .filter((pl.col('recency') >= pl.col('m_gap') - 3) & (pl.col('recency') <= pl.col('m_gap') + 5))\
            .select(['customer_id', 'item_id']).unique()
        cands['repl'] = due_items

        # Combine
        all_cands = pl.concat([df for df in cands.values() if df.height > 0]).unique()
        
        # Hard Filter (Geography + Price)
        user_prof = hist_s.group_by('customer_id').agg([
            pl.col('location').mode().first().alias('loc'),
            pl.col('price').mean().alias('avg_p')
        ])
        item_loc_flat = self.item_locs.explode('item_hubs').rename({'item_hubs': 'loc'})
        f = all_cands.join(user_prof, on='customer_id', how='left')\
            .join(item_loc_flat, on=['item_id', 'loc'], how='inner')\
            .join(self.item_prcs, on='item_id', how='left')\
            .filter((pl.col('item_p') <= pl.col('avg_p') * 10) | (pl.col('avg_p').is_null()))\
            .select(['customer_id', 'item_id'])
            
        return f


=== V9 RETRIEVER: MULTI-SOURCE FUNNEL ===


In [4]:
def create_dataset_v9(history_df, truth_df, items_df, sample_users=None, n_negatives=100):
    if sample_users:
        valid_u = history_df['customer_id'].unique().shuffle(seed=SEED).head(sample_users).to_list()
    else:
        valid_u = history_df['customer_id'].unique().to_list()
    
    retriever = V9Retriever(history_df, items_df)
    filtered_cands = retriever.get_candidates(valid_u)
    
    if truth_df is not None:
        truth = truth_df.filter(pl.col('customer_id').is_in(valid_u)).select(['customer_id', 'item_id']).unique()
        ds = filtered_cands.join(truth.with_columns(pl.lit(1).cast(pl.Int8).alias('target')), on=['customer_id', 'item_id'], how='left').fill_null(0)
        missed = truth.join(filtered_cands, on=['customer_id', 'item_id'], how='anti').with_columns(pl.lit(1).cast(pl.Int8).alias('target'))
        ds = pl.concat([ds, missed]).unique(subset=['customer_id', 'item_id'])
        if n_negatives:
            pos = ds.filter(pl.col('target') == 1)
            neg = ds.filter(pl.col('target') == 0).sample(fraction=1.0, shuffle=True, seed=SEED).group_by('customer_id').head(n_negatives)
            ds = pl.concat([pos, neg]).sort(['customer_id', 'target'], descending=[False, True])
    else:
        ds = filtered_cands
    
    # --- Feature Engineering (Idea Scatter) ---
    max_ts = history_df['event_ts'].max()
    
    # User Profile (Idea 8, 41, 42)
    u_prof = history_df.group_by('customer_id').agg([
        pl.col('item_id').n_unique().alias('u_unique_items'),
        pl.col('quantity').sum().alias('u_total_qty'),
        (max_ts - pl.col('event_ts').min()).dt.total_days().alias('u_tenure_days'),
        pl.col('price').mean().alias('u_avg_price'),
        (pl.col('item_id').n_unique() / pl.col('quantity').sum().clip(1)).alias('u_exploration_ratio')
    ])
    
    # Item Profile (Idea 17, 23, 30)
    i_prof = history_df.group_by('item_id').agg([
        pl.col('customer_id').n_unique().alias('i_unique_users'),
        pl.col('quantity').sum().alias('i_total_qty'),
        pl.col('location').n_unique().alias('i_hubs_count')
    ])
    
    # User-Item Interaction (Idea 44)
    ui_hist = history_df.filter(pl.col('customer_id').is_in(valid_u)).group_by(['customer_id', 'item_id']).agg([
        pl.col('quantity').sum().alias('ui_total_qty'),
        pl.col('event_ts').max().alias('ui_last_buy_ts')
    ]).with_columns((max_ts - pl.col('ui_last_buy_ts')).dt.total_days().alias('ui_recency_days'))
    
    # Momentum (Idea 23)
    vol_7d = history_df.filter(pl.col('event_ts') >= max_ts - pl.duration(days=7)).group_by('item_id').len().rename({'len': 'v7'})
    vol_21d = history_df.filter(pl.col('event_ts') >= max_ts - pl.duration(days=21)).group_by('item_id').len().rename({'len': 'v21'})
    momentum = vol_7d.join(vol_21d, on='item_id', how='left').with_columns((pl.col('v7') / (pl.col('v21') / 3.0 + 1)).alias('item_momentum'))
    
    # Category Affinity (Idea 42)
    u_cat_pref = history_df.join(items_df.select(['item_id', 'category_l1']), on='item_id')\
        .group_by(['customer_id', 'category_l1']).len()\
        .with_columns((pl.col('len') / pl.col('len').sum().over('customer_id')).alias('u_cat_affinity'))
    
    ds = ds.join(u_prof, on='customer_id', how='left')
    ds = ds.join(i_prof, on='item_id', how='left')
    ds = ds.join(ui_hist.drop('ui_last_buy_ts'), on=['customer_id', 'item_id'], how='left')
    ds = ds.join(items_df.select(['item_id', 'item_age_proxy'] + cat_cols + [f'{c}_id' for c in cat_cols]), on='item_id', how='left')
    ds = ds.join(momentum.select(['item_id', 'item_momentum']), on='item_id', how='left')
    ds = ds.join(u_cat_pref.select(['customer_id', 'category_l1', 'u_cat_affinity']), on=['customer_id', 'category_l1'], how='left')
    
    # Safe Fill
    return ds.with_columns([
        pl.col(pl.Utf8).fill_null('Unknown'),
        pl.col(pl.Float32, pl.Float64, pl.Int32, pl.Int64, pl.Int8).fill_null(0)
    ])


In [5]:
print("=== V9 ROBUST CROSS-VALIDATION ===")
def get_fold(train_end, val_m):
    h = df_raw.filter(pl.col('month') <= train_end)
    t = df_raw.filter(pl.col('month') == val_m)
    return create_dataset_v9(h, t, items_df, sample_users=60000, n_negatives=120)

# 3-Fold CV for robust tuning
fold1 = get_fold(8, 9)   # Train thru Aug, Val Sep
fold2 = get_fold(9, 10)  # Train thru Sep, Val Oct
fold3 = get_fold(10, 11) # Train thru Oct, Val Nov
# Final validation on December
test_set = create_dataset_v9(df_raw.filter(pl.col('month') <= 11), df_raw.filter(pl.col('month') == 12), items_df, sample_users=35000, n_negatives=None)

print("=== OPTUNA GPU TRAINING ===")
cat_feat_ids = [f'{c}_id' for c in cat_cols]
all_feats = [
    'u_unique_items', 'u_total_qty', 'u_tenure_days', 'u_avg_price', 'u_exploration_ratio',
    'i_unique_users', 'i_total_qty', 'i_hubs_count',
    'ui_total_qty', 'ui_recency_days', 'item_momentum', 'item_age_proxy', 'u_cat_affinity'
] + cat_feat_ids

def prep_lgb(df):
    p = df.to_pandas()
    # Keep as integers, LightGBM will handle them via categorical_feature parameter
    return p[all_feats], p['target'], p.groupby('customer_id').size().values

X1, y1, g1 = prep_lgb(fold1)
X2, y2, g2 = prep_lgb(fold2)
X3, y3, g3 = prep_lgb(fold3)

def objective(trial):
    param = {
        'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [10], 'verbosity': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 63, 1023), # Increased for GPU
        'max_depth': trial.suggest_int('max_depth', 8, 20),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 50, 500),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.9),
        'max_bin': 255, # Fixed limit for GPU compatibility
        'device': 'gpu', # THE KEY: GPU ACCELERATION
        'gpu_platform_id': 0, 'gpu_device_id': 0,
        'random_state': SEED
    }
    # Train on F1+F2, Validate on F3
    X_train = pd.concat([X1, X2])
    y_train = pd.concat([y1, y2])
    g_train = np.concatenate([g1, g2])
    
    dtrain = lgb.Dataset(X_train, y_train, group=g_train, categorical_feature=cat_feat_ids)
    dval = lgb.Dataset(X3, y3, group=g3, reference=dtrain, categorical_feature=cat_feat_ids)
    m = lgb.train(param, dtrain, valid_sets=[dval], num_boost_round=600, callbacks=[lgb.early_stopping(50)])
    score = m.best_score['valid_0']['ndcg@10']
    
    # Explicit Cleanup for RAM Safety
    del m, dtrain, dval, X_train, y_train, g_train
    gc.collect()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40)
best_params = study.best_params
print("Best Params (V9):", best_params)

# Final Training on all 3 folds
X_final = pd.concat([X1, X2, X3])
y_final = pd.concat([y1, y2, y3])
g_final = np.concatenate([g1, g2, g3])
d_final = lgb.Dataset(X_final, y_final, group=g_final, categorical_feature=cat_feat_ids)
best_params.update({'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [10], 'device': 'gpu', 'max_bin': 255})
lgb_m = lgb.train(best_params, d_final, num_boost_round=1200)


=== V9 ROBUST CROSS-VALIDATION ===
 Building Popularity & Hub Indexes...
 Building SVD (160) & I2I Matrices...
 Building Size Ladder...
 Building Replenishment Cycles...
 Building Popularity & Hub Indexes...
 Building SVD (160) & I2I Matrices...
 Building Size Ladder...
 Building Replenishment Cycles...
 Building Popularity & Hub Indexes...
 Building SVD (160) & I2I Matrices...
 Building Size Ladder...
 Building Replenishment Cycles...
 Building Popularity & Hub Indexes...
 Building SVD (160) & I2I Matrices...
 Building Size Ladder...
 Building Replenishment Cycles...
=== OPTUNA GPU TRAINING ===


[I 2026-05-16 14:21:55,788] A new study created in memory with name: no-name-0fec9bfc-ff0e-4af5-97c1-86978f3bc12c
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[33]	valid_0's ndcg@10: 0.896232


[I 2026-05-16 14:25:03,628] Trial 0 finished with value: 0.8962324667453929 and parameters: {'learning_rate': 0.02243920528741162, 'num_leaves': 696, 'max_depth': 10, 'min_data_in_leaf': 271, 'colsample_bytree': 0.4872844591549407}. Best is trial 0 with value: 0.8962324667453929.


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[13]	valid_0's ndcg@10: 0.895702


...

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[12]	valid_0's ndcg@10: 0.896219


[I 2026-05-16 16:37:23,297] Trial 38 finished with value: 0.896219394945847 and parameters: {'learning_rate': 0.02919638934453163, 'num_leaves': 932, 'max_depth': 19, 'min_data_in_leaf': 343, 'colsample_bytree': 0.6727697517264893}. Best is trial 14 with value: 0.8988442630439961.


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[15]	valid_0's ndcg@10: 0.898735


[I 2026-05-16 16:40:01,186] Trial 39 finished with value: 0.8987346121972416 and parameters: {'learning_rate': 0.08593880544470942, 'num_leaves': 651, 'max_depth': 14, 'min_data_in_leaf': 196, 'colsample_bytree': 0.7244227872229159}. Best is trial 14 with value: 0.8988442630439961.


Best Params (V9): {'learning_rate': 0.060850422749801625, 'num_leaves': 1006, 'max_depth': 16, 'min_data_in_leaf': 245, 'colsample_bytree': 0.6873328506397279}


In [6]:
print("=== FINAL EVALUATION (MONTH 12) ===")
X_ts, y_ts, _ = prep_lgb(test_set)
test_set = test_set.with_columns(pl.Series(name='pred', values=lgb_m.predict(X_ts)))

def evaluate(model_col):
    top10 = test_set.sort(['customer_id', model_col], descending=[False, True]).group_by('customer_id', maintain_order=True).head(10)
    truth_map = df_raw.filter(pl.col('month') == 12).filter(pl.col('customer_id').is_in(top10['customer_id'].unique().to_list())).group_by('customer_id').agg(pl.col('item_id'))
    truth_dict = {row[0]: set(row[1]) for row in truth_map.iter_rows()}
    pred_dict = {row[0]: list(row[1]) for row in top10.group_by('customer_id').agg(pl.col('item_id')).iter_rows()}
    h, m, p = 0, 0.0, 0.0
    for uid, truth in truth_dict.items():
        preds = pred_dict.get(uid, [])
        hits = [pr for pr in preds if pr in truth]
        h += len(hits); p += len(hits)/10.0
        for i, pr in enumerate(preds):
            if pr in truth: m += 1.0/(i+1); break
    n = max(1, len(truth_dict))
    return {'Hits': h, 'Precision@10': p/n, 'MRR': m/n}

print(evaluate('pred'))


=== FINAL EVALUATION (MONTH 12) ===
{'Hits': 15986, 'Precision@10': 0.19585885812300466, 'MRR': 0.6346552743453004}
